# Preconditioned Crank-Nicolson Langevin (MALA) Tutorial

This tutorial demonstrates `ls_bayesian`'s function-space MALA sampler, `MALAAlgorithm`
(`ls_bayesian.mcmc.algorithms.mala`) -- the pCNL variant of MALA (Cotter, Roberts, Stuart, White,
2013) that uses the target's gradient to drive a Langevin proposal, without any preconditioning
(see the `pmala` tutorial for that generalization). As in the `pcn` tutorial, we use a
linear-Gaussian toy inverse problem whose posterior is known in closed form, so every sampler run
can be checked against an exact reference.

## Mathematical Formulation

`MALAAlgorithm` keeps the target's potential $\Phi$ relative to the reference measure $\mu_0 =
\mathcal N(\bar u, C)$ throughout: unlike `PCNAlgorithm`, it has no Pinski-et-al.-style
generalized-approximation counterpart, so `model.reference` is the only Gaussian it reads (see
`MALAAlgorithm`'s docstring for why re-expressing $\Phi$ relative to a different Gaussian, as pCN
does, would additionally require the *gradient* of the correction potential).

Given the current state $u$, the proposal is

$$
v = \bar u + \frac{2-\delta}{2+\delta}(u - \bar u) - \frac{2\delta}{2+\delta} C\nabla\Phi(u) +
\frac{\sqrt{8\delta}}{2+\delta}\, w, \qquad w \sim \mathcal N(0, C),
$$

with acceptance probability $\alpha(u,v) = 1 \wedge \exp(\varrho(u,v) - \varrho(v,u))$, where

$$
\varrho(u,v) = \Phi(u) + \frac{1}{2}(v-u,\nabla\Phi(u)) + \frac{\delta}{4}(u+v-2\bar
u,\nabla\Phi(u)) + \frac{\delta}{4}(\nabla\Phi(u), C\nabla\Phi(u)).
$$

Unlike pCN's step width $\delta \in (0,1)$, MALA's $\delta > 0$ is unbounded above -- smaller
values increase acceptance, larger values explore faster until the linearized drift becomes too
aggressive and acceptance collapses.

References:

- Cotter, Roberts, Stuart, White (2013). *MCMC Methods for Functions: Modifying Old Algorithms to
  Make Them Faster.* Statistical Science 28(3).

## Imports and Configuration

We import NumPy for the linear algebra, Matplotlib (plus `matplotlib.patches.Ellipse` for covariance ellipses) for the diagnostic plots, `scipy.stats.norm` for the analytic 1D marginal density, `typing.override` for the measure/target implementations, and the `algorithms.mala`, `measures`, `sampler`, and `output` modules of `ls_bayesian`'s `mcmc` subpackage. All randomness is seeded for reproducibility.

In [ ]:
from typing import override

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse
from scipy.stats import norm

from ls_bayesian.mcmc import output as mcmc_output
from ls_bayesian.mcmc.algorithms.mala import MALAAlgorithm
from ls_bayesian.mcmc.measures import DifferentiableTargetMeasure, GaussianMeasure
from ls_bayesian.mcmc.model import MCMCModel
from ls_bayesian.mcmc.sampler import Sampler, SamplerSettings
from ls_bayesian.mcmc.storage import NumpyStorage

rng = np.random.default_rng(0)
STATE_DIM = 4

## A Linear-Gaussian Test Problem

We reuse the same test problem as the `pcn`/`pmala` tutorials: a quadratic potential
$\Phi(u) = \frac{1}{2}(u-a)^T H (u-a)$ combined with a centered Gaussian reference $\mu_0 =
\mathcal N(0, C)$, giving a Gaussian target with precision $H + C^{-1}$ and mean
$(H+C^{-1})^{-1} H a$.

In [ ]:
def random_spd_matrix(rng: np.random.Generator, dim: int) -> np.ndarray:
    """Return a random symmetric positive-definite matrix of shape (dim, dim)."""
    factor = rng.random((dim, dim))
    return factor @ factor.T + dim * np.eye(dim)


hessian = random_spd_matrix(rng, STATE_DIM)
prior_covariance = random_spd_matrix(rng, STATE_DIM)
minimizer = rng.standard_normal(STATE_DIM)

prior_precision = np.linalg.inv(prior_covariance)
posterior_precision = hessian + prior_precision
posterior_covariance = np.linalg.inv(posterior_precision)
posterior_mean = posterior_covariance @ (hessian @ minimizer)
posterior_standard_deviation = np.sqrt(np.diag(posterior_covariance))

print("Analytic posterior mean:", posterior_mean)
print("Analytic posterior marginal std :", posterior_standard_deviation)

## Implementing the Target and the Reference Measure

`MALAAlgorithm` takes one `MCMCModel`: `target` (a `DifferentiableTargetMeasure` for $\Phi$ and
$\nabla\Phi$) and `reference` (a `GaussianMeasure` for $\mu_0$'s mean/covariance/precision).
`model.approximation` is left out entirely -- `MALAAlgorithm` would only ignore it anyway.

In [ ]:
class QuadraticTargetMeasure(DifferentiableTargetMeasure):
    """Quadratic potential Phi(u) = 1/2 (u-a)^T H (u-a), exact gradient H(u-a)."""

    def __init__(self, matrix: np.ndarray, minimizer: np.ndarray) -> None:
        self.matrix = matrix
        self.minimizer = minimizer

    @override
    def evaluate_potential(self, state: np.ndarray) -> float:
        difference = state - self.minimizer
        return float(0.5 * difference @ self.matrix @ difference)

    @override
    def evaluate_gradient(self, state: np.ndarray) -> np.ndarray:
        return self.matrix @ (state - self.minimizer)


class DenseGaussianMeasure(GaussianMeasure):
    """Centered Gaussian measure N(0, covariance), usable as mu_0 (`reference`)."""

    def __init__(self, covariance_matrix: np.ndarray) -> None:
        self.covariance_matrix = covariance_matrix
        self.precision_matrix = np.linalg.inv(covariance_matrix)
        self.covariance_factor = np.linalg.cholesky(covariance_matrix)

    @property
    @override
    def mean(self) -> np.ndarray:
        return np.zeros(self.covariance_matrix.shape[0])

    @property
    @override
    def random_vector_size(self) -> int:
        return self.covariance_factor.shape[1]

    @override
    def apply_covariance_factorization(self, random_vector: np.ndarray) -> np.ndarray:
        return self.covariance_factor @ random_vector

    @override
    def apply_covariance_operator(self, vector: np.ndarray) -> np.ndarray:
        return self.covariance_matrix @ vector

    @override
    def apply_precision_operator(self, vector: np.ndarray) -> np.ndarray:
        return self.precision_matrix @ vector


target = QuadraticTargetMeasure(hessian, minimizer)
reference_measure = DenseGaussianMeasure(prior_covariance)

## Running the Sampler

`Sampler` drives `MALAAlgorithm.compute_step` for a fixed number of samples, dispatching each new
state to `NumpyStorage` and to an `MCMCOutput` tracking the running-mean acceptance rate. Because
$H$ and $C^{-1}$ are unrelated random matrices here, the posterior's geometry is poorly matched by
$C$, so we start with a small step width.

In [ ]:
BURN_IN = 2000
NUM_SAMPLES = 20_000


def run_mala_chain(step_width: float, seed: int) -> tuple[np.ndarray, float]:
    """Run MALAAlgorithm for BURN_IN + NUM_SAMPLES steps and return the post-burn-in samples and
    the final running-mean acceptance rate."""
    model = MCMCModel(target=target, reference=reference_measure)
    algorithm = MALAAlgorithm(model, step_width)
    acceptance_output = mcmc_output.build(
        mcmc_output.AcceptanceQoI(), mcmc_output.RunningMeanStatistic()
    )
    storage = NumpyStorage()
    sampler = Sampler(algorithm, storage=storage, outputs=[acceptance_output])
    settings = SamplerSettings(
        num_samples=BURN_IN + NUM_SAMPLES, log_interval=BURN_IN + NUM_SAMPLES
    )
    sampler.run(posterior_mean.copy(), settings, seed=seed)
    return storage.values[BURN_IN:], acceptance_output.value


samples, acceptance_rate = run_mala_chain(step_width=0.02, seed=1)
print(f"MALA acceptance rate (step_width=0.02): {acceptance_rate:.3f}")

## Verifying the Recovered Posterior Moments

The empirical mean and standard deviation of the post-burn-in samples should match the analytic
posterior moments computed above, up to Monte Carlo error.

In [ ]:
def report_moment_recovery(samples: np.ndarray, label: str) -> None:
    """Print the sample mean/std next to the analytic reference, normalized by the analytic
    standard deviation so the comparison is scale-free across coordinates."""
    sample_mean = samples.mean(axis=0)
    sample_std = samples.std(axis=0, ddof=1)
    normalized_mean_error = (sample_mean - posterior_mean) / posterior_standard_deviation
    print(f"{label}:")
    print("  normalized mean error:", normalized_mean_error)
    print("  sample std / analytic std:", sample_std / posterior_standard_deviation)
    assert np.all(np.abs(normalized_mean_error) < 0.5)
    assert np.all(np.abs(sample_std / posterior_standard_deviation - 1.0) < 0.3)


report_moment_recovery(samples, "MALA")

## Trace Plot

Plotting each state component against sample index is the standard first check for a chain's mixing behavior: a well-mixing chain looks like stationary noise around the analytic mean (dashed line), with no visible drift or long excursions.

In [ ]:
fig, axes = plt.subplots(STATE_DIM, 1, figsize=(8, 2 * STATE_DIM), sharex=True)
for component, ax in enumerate(axes):
    ax.plot(samples[:, component], linewidth=0.5)
    ax.axhline(posterior_mean[component], color="black", linestyle="--", linewidth=1)
    ax.set_ylabel(f"$u_{component}$")
axes[-1].set_xlabel("post-burn-in sample index")
fig.suptitle("MALA: trace plot")
fig.tight_layout()

## 1D Marginals vs. Analytical

Each component's empirical histogram should match its analytic marginal $\mathcal N(\bar u_i, \Sigma_{ii})$ (the posterior mean/std computed above), since every marginal of a Gaussian is itself Gaussian.

In [ ]:
fig, axes = plt.subplots(1, STATE_DIM, figsize=(4 * STATE_DIM, 3))
for component, ax in enumerate(axes):
    ax.hist(samples[:, component], bins=50, density=True, alpha=0.6, label="samples")
    grid = np.linspace(*ax.get_xlim(), 200)
    analytic_density = norm.pdf(
        grid, posterior_mean[component], posterior_standard_deviation[component]
    )
    ax.plot(grid, analytic_density, color="black", label="analytic")
    ax.set_xlabel(f"$u_{component}$")
axes[0].set_ylabel("density")
axes[0].legend()
fig.suptitle("MALA: 1D marginals vs. analytic Gaussian")
fig.tight_layout()

## Pairwise 2D Marginals vs. Analytical

A scatter-plot matrix of every pair of components, each overlaid with the $1\sigma$/$2\sigma$ covariance ellipses of the analytic bivariate Gaussian marginal for that pair (the corresponding $2\times 2$ block of the analytic posterior covariance) -- the empirical point cloud should hug those ellipses.

In [ ]:
def add_covariance_ellipses(ax, mean_pair, covariance_pair, n_std_values=(1, 2)) -> None:
    """Draw n_std_values-sigma covariance ellipses of a 2D Gaussian with the given mean/covariance."""
    eigenvalues, eigenvectors = np.linalg.eigh(covariance_pair)
    angle = np.degrees(np.arctan2(eigenvectors[1, -1], eigenvectors[0, -1]))
    for n_std in n_std_values:
        width, height = 2 * n_std * np.sqrt(eigenvalues[::-1])
        ax.add_patch(
            Ellipse(mean_pair, width, height, angle=angle, fill=False, edgecolor="black", lw=1)
        )


fig, axes = plt.subplots(STATE_DIM, STATE_DIM, figsize=(3 * STATE_DIM, 3 * STATE_DIM))
for row in range(STATE_DIM):
    for col in range(STATE_DIM):
        ax = axes[row, col]
        if row <= col:
            ax.axis("off")
            continue
        ax.scatter(samples[:, col], samples[:, row], s=2, alpha=0.15)
        add_covariance_ellipses(
            ax,
            posterior_mean[[col, row]],
            posterior_covariance[np.ix_([col, row], [col, row])],
        )
        if row == STATE_DIM - 1:
            ax.set_xlabel(f"$u_{col}$")
        if col == 0:
            ax.set_ylabel(f"$u_{row}$")
fig.suptitle("MALA: pairwise 2D marginals vs. analytic covariance ellipses")
fig.tight_layout()

## Step-Width Sweep

Sweeping $\delta$ over several orders of magnitude makes the acceptance/exploration trade-off
concrete: acceptance falls off sharply once the linearized drift's scale starts to compete with
the posterior's own, comparatively small, geometry.

In [ ]:
def sweep_acceptance_rate(step_widths: np.ndarray) -> np.ndarray:
    """Return the final running-mean acceptance rate for each step width in `step_widths`, from a
    short chain (fine for an acceptance-rate estimate, unlike the moment-recovery run above)."""
    acceptance_rates = np.empty_like(step_widths)
    for index, step_width in enumerate(step_widths):
        _, acceptance_rates[index] = run_mala_chain(float(step_width), seed=1)
    return acceptance_rates


sweep_step_widths = np.array([0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 4.0, 8.0])
acceptance_sweep = sweep_acceptance_rate(sweep_step_widths)

fig, ax = plt.subplots()
ax.plot(sweep_step_widths, acceptance_sweep, marker="o")
ax.set_xscale("log")
ax.set_xlabel(r"step width $\delta$")
ax.set_ylabel("acceptance rate")
ax.set_ylim(-0.05, 1.05)
ax.set_title("MALA acceptance rate vs. step width")
fig.tight_layout()

## Summary

- `MALAAlgorithm` keeps $\Phi$ relative to the target's true reference measure $\mu_0$ throughout,
  using the target's gradient $\nabla\Phi$ to drive a Langevin proposal via `model.reference`
  alone -- no `model.approximation` is needed or read.
- The recovered posterior mean and standard deviation matched the analytic reference to within
  Monte Carlo error.
- As with `pmala`'s plain-MALA-equivalent baseline, acceptance collapses quickly as $\delta$ grows
  once the posterior's geometry departs from the reference's -- exactly the gap that
  `PMALAAlgorithm`'s preconditioning (see the `pmala` tutorial) closes.